# HDB Resale Flat Prices — ETL Pipeline

## Project Structure

```
DEA/
├── raw_data/ResaleFlatPrices/          # [GROUP 1: RAW] 3 source CSVs
│   ├── 2000-Feb2012 (Approval).csv
│   ├── Mar2012-Dec2014 (Registration).csv
│   └── Jan2015-Dec2016 (Registration).csv
│
├── manipulated_data/                   # Pipeline output files
│   ├── cleaned_dataset.csv             # [GROUP 2: CLEANED]
│   ├── transformed_dataset.csv         # [GROUP 3: TRANSFORMED]
│   ├── failed_records.csv              # [GROUP 4: FAILED]
│   └── hashed_dataset.csv              # [GROUP 5: HASHED]
│
├── profiling_outputs/                  # Reports & visualisations
│   ├── profiling_summary.txt
│   ├── profiling_visualisations.png
│   ├── validation_rules.json
│   └── anomaly_summary.json
│
└── HDB_analysis.ipynb                  # This notebook
```

## Tasks

| # | Task | Description |
|---|------|-------------|
| 1 | Combine | Load 3 CSVs, align schemas, filter Jan 2012–Dec 2016 |
| 2 | Profile | Generate text report + 6-panel matplotlib figure |
| 3 | Validate | Derive per-field validation rules from data stats |
| 4 | Lease | Recompute remaining lease (99-year, as of today) |
| 5 | Dedup | Composite-key dedup (keep max price per key group) |
| 6 | Anomaly | Detect anomalous prices (IQR + z-score heuristics) |
| 7 | Checks | Benford's Law, Isolation Forest, YoY price jumps |
| 8 | Hash + Export | Build resale identifier, SHA-256 hash, export 5 groups |

In [1]:
# Standard library imports
import json
import os
import re
from datetime import date, datetime
from pathlib import Path

# Third-party imports
import numpy as np
import pandas as pd

# ---------------------------------------------------------------------------
# Configuration
# ---------------------------------------------------------------------------
# Set matplotlib config directory to a local cache to avoid permission issues
# in restricted environments.
os.environ["MPLCONFIGDIR"] = str(Path.cwd() / ".mpl_cache")

TODAY: date = date.today()

In [3]:
"""
data_config — Source file metadata, path configuration, and lease-parsing
utilities for the HDB resale flat price analysis pipeline.
"""

from __future__ import annotations

import re
from dataclasses import dataclass, field
from pathlib import Path
from typing import Union

# ── Constants ──────────────────────────────────────────────────────────────

MONTHS_PER_YEAR: int = 12

# Regex: "61 years 04 months" → groups (61, 04)
LEASE_PARSE_PATTERN: re.Pattern[str] = re.compile(
    r"(\d+)\s*years?\s*(?:(\d+)\s*months?)?",
)

# Date boundaries for the analysis window
START_MONTH: str = "2012-01"
END_MONTH: str = "2016-12"

# Type alias for values that ``parse_lease`` can accept
LeaseValue = Union[str, int, float, None]


# ── Data classes ───────────────────────────────────────────────────────────

@dataclass(frozen=True, slots=True)
class SourceFile:
    """Metadata for a single raw CSV source file.

    Attributes:
        label:               Human-readable descriptor for the date range /
                             pricing basis (e.g. "Mar2012-Dec2014 (Registration)").
        filename:            Name of the CSV file inside the raw-data directory.
        has_remaining_lease: Whether this file includes a ``remaining_lease``
                             column that needs parsing.
    """

    label: str
    filename: str
    has_remaining_lease: bool


DEFAULT_SOURCES: list[SourceFile] = [
    SourceFile(
        label="2000-Feb2012 (Approval)",
        filename="Resale Flat Prices (Based on Approval Date), 2000 - Feb 2012.csv",
        has_remaining_lease=False,
    ),
    SourceFile(
        label="Mar2012-Dec2014 (Registration)",
        filename="Resale Flat Prices (Based on Registration Date), From Mar 2012 to Dec 2014.csv",
        has_remaining_lease=False,
    ),
    SourceFile(
        label="Jan2015-Dec2016 (Registration)",
        filename="Resale Flat Prices (Based on Registration Date), From Jan 2015 to Dec 2016.csv",
        has_remaining_lease=True,
    ),
]


@dataclass
class Config:
    """Central path configuration for the analysis pipeline.

    All paths are derived from ``root``, which defaults to the current
    working directory.  **Note:** the module-level ``CFG`` singleton is
    created at import time, so ``root`` will reflect whatever directory
    is current when the module is first imported.
    """

    root: Path = field(default_factory=Path.cwd)
    sources: list[SourceFile] = field(default_factory=lambda: list(DEFAULT_SOURCES))

    # Derived paths — populated by __post_init__
    raw_dir: Path = field(init=False)
    manipulated_dir: Path = field(init=False)
    report_dir: Path = field(init=False)
    master_path: Path = field(init=False)
    profile_path: Path = field(init=False)
    rules_path: Path = field(init=False)
    plot_path: Path = field(init=False)

    def __post_init__(self) -> None:
        self.raw_dir = self.root / "raw_data" / "ResaleFlatPrices"
        self.manipulated_dir = self.root / "manipulated_data"
        self.report_dir = self.root / "profiling_outputs"
        self.master_path = self.manipulated_dir / "master_dataset.csv"
        self.profile_path = self.report_dir / "profiling_summary.txt"
        self.rules_path = self.report_dir / "validation_rules.json"
        self.plot_path = self.report_dir / "profiling_visualisations.png"

    def ensure_directories(self) -> None:
        """Create output directories if they do not already exist."""
        for directory in (self.manipulated_dir, self.report_dir):
            directory.mkdir(parents=True, exist_ok=True)


# ── Parsing helper ─────────────────────────────────────────────────────────

def parse_lease(value: LeaseValue) -> float:
    """Convert a remaining-lease value to fractional years.

    Parameters
    ----------
    value : str, int, float, or None
        The raw lease value.  Accepts numeric types (returned as-is),
        human-readable strings like ``"61 years 04 months"``, or
        ``None`` / ``NaN``.

    Returns
    -------
    float
        Lease duration in years, or ``NaN`` if the input is missing or
        cannot be parsed.

    Examples
    --------
    >>> parse_lease("61 years 04 months")
    61.333333...
    >>> parse_lease(99)
    99.0
    >>> parse_lease(None)
    nan
    """
    if pd.isna(value):
        return np.nan
    if isinstance(value, (int, float)):
        return float(value)

    match = LEASE_PARSE_PATTERN.match(str(value).strip().lower())
    if not match:
        return np.nan

    years = int(match.group(1))
    months = int(match.group(2)) if match.group(2) else 0
    return years + months / MONTHS_PER_YEAR


# ── Module-level singleton ─────────────────────────────────────────────────
# Instantiated at import time — root is locked to the CWD at that moment.
CFG = Config()

## 1. Combine

In [4]:
def _summarise_dataframe(df: pd.DataFrame, label: str = "DataFrame") -> None:
    """Print a structural overview and head/tail preview of *df*.

    Parameters
    ----------
    label : str
        A human-readable name shown in the header of the summary.
    """
    separator = "-" * 60
    print(f"\n{separator}")
    print(f"  {label}")
    print(separator)
    print(f"  Shape   : {df.shape[0]:,} rows x {df.shape[1]} columns")
    print(f"  Columns : {', '.join(df.columns)}")
    print(f"  DTypes  :\n{df.dtypes.to_string()}\n")
    print(f"  Nulls   :\n{df.isnull().sum().to_string()}\n")
    print(f"  Head (5 rows) {'-' * 43}")
    print(df.head().to_string(index=False))
    print(f"\n  Tail (5 rows) {'-' * 43}")
    print(df.tail().to_string(index=False))
    print(separator, "\n")


def task1_combine() -> pd.DataFrame:
    """Load all source CSVs, align schemas, filter to the analysis date
    range, and merge into a single master dataset.

    Returns
    -------
    pd.DataFrame
        The combined, cleaned master dataset.

    Raises
    ------
    FileNotFoundError
        If any source CSV is missing from ``CFG.raw_dir``.
    ValueError
        If no rows remain after filtering.
    """
    frames: list[pd.DataFrame] = []

    for source in CFG.sources:
        path = CFG.raw_dir / source.filename
        if not path.exists():
            raise FileNotFoundError(f"Source file not found: {path}")

        df = pd.read_csv(path)
        df["source"] = source.label

        if not source.has_remaining_lease:
            df["remaining_lease"] = np.nan

        df["month"] = pd.to_datetime(df["month"], errors="coerce")
        df = df.loc[
            (df["month"] >= START_MONTH) & (df["month"] <= END_MONTH)
        ]

        print(f"  {source.label}: {len(df):,} rows retained")
        frames.append(df)

    if not frames:
        raise ValueError(
            "No data frames to combine -- check CFG.sources and date range."
        )

    master = (
        pd.concat(frames, ignore_index=True)
        .assign(
            remaining_lease=lambda d: d["remaining_lease"].apply(parse_lease),
            flat_model=lambda d: d["flat_model"].str.title(),
        )
        .sort_values("month", ignore_index=True)
    )

    CFG.ensure_directories()
    master.to_csv(CFG.master_path, index=False)

    print(
        f"\n  Master saved: {len(master):,} rows x {master.shape[1]} cols "
        f"-> {CFG.master_path}"
    )
    _summarise_dataframe(master, label="Master Dataset")

    return master


master = task1_combine()

  2000-Feb2012 (Approval): 3,188 rows retained
  Mar2012-Dec2014 (Registration): 52,203 rows retained
  Jan2015-Dec2016 (Registration): 37,153 rows retained

  Master saved: 92,544 rows x 12 cols -> /Users/john.yap/Desktop/DEA/manipulated_data/master_dataset.csv

------------------------------------------------------------
  Master Dataset
------------------------------------------------------------
  Shape   : 92,544 rows x 12 columns
  Columns : month, town, flat_type, block, street_name, storey_range, floor_area_sqm, flat_model, lease_commence_date, resale_price, source, remaining_lease
  DTypes  :
month                  datetime64[us]
town                              str
flat_type                         str
block                             str
street_name                       str
storey_range                      str
floor_area_sqm                float64
flat_model                        str
lease_commence_date             int64
resale_price                  float64
source     

## 2. Profile

In [5]:
# ── Report helpers ──────────────────────────────────────────────────────────

SEPARATOR = "-" * 65


def _heading(lines: list[str], title: str) -> None:
    """Add a section divider with a centred title."""
    lines.extend([SEPARATOR, f"  {title}", SEPARATOR])


def _pct(part: int, total: int) -> float:
    """Safe percentage calculation (avoids division by zero)."""
    return part / total * 100 if total else 0.0


# ── Report sections ─────────────────────────────────────────────────────────

def _overview_section(df: pd.DataFrame, lines: list[str]) -> None:
    """Dataset-level summary: shape, memory, date range, cardinalities."""
    _heading(lines, "Dataset Overview")
    lines.extend([
        f"  Total rows              : {len(df):,}",
        f"  Total columns           : {len(df.columns)}",
        f"  Memory (approx)         : {df.memory_usage(deep=True).sum() / 1024**2:.2f} MB",
        f"  Date range              : {df['month'].min():%Y-%m}  to  {df['month'].max():%Y-%m}",
        f"  Distinct towns          : {df['town'].nunique()}",
        f"  Distinct flat types     : {df['flat_type'].nunique()}",
        f"  Distinct flat models    : {df['flat_model'].nunique()}",
        f"  Distinct storey ranges  : {df['storey_range'].nunique()}",
        "",
    ])


def _column_profile_section(df: pd.DataFrame, lines: list[str]) -> None:
    """One row per column: data type, non-null count, null %, unique values."""
    _heading(lines, "Column Profiles")
    null_counts = df.isnull().sum()
    null_pcts = null_counts / len(df) * 100
    n = len(df)

    header = (
        f"  {'Column':<25} {'Type':<16} {'Non-Null':>10} {'Null%':>7} {'Unique':>8}"
    )
    lines.extend([header, "  " + "-" * (len(header) - 2)])

    for col in df.columns:
        nulls = null_counts[col]
        lines.append(
            f"  {col:<25} {str(df[col].dtype):<16} "
            f"{n - nulls:>10,} {_pct(nulls, n):>6.1f}% "
            f"{df[col].nunique():>8,}"
        )
    lines.append("")


def _numeric_stats_section(df: pd.DataFrame, lines: list[str]) -> None:
    """Descriptive statistics for all numeric columns."""
    numeric_cols = df.select_dtypes(include=[np.number]).columns.tolist()
    if not numeric_cols:
        return

    _heading(lines, "Numeric Descriptive Statistics")
    desc = df[numeric_cols].describe(
        percentiles=[0.01, 0.05, 0.25, 0.5, 0.75, 0.95, 0.99]
    )
    col_width = max(len(c) for c in desc.columns) + 2

    lines.append(
        f"  {'Statistic':<12} " + "".join(f"{c:<{col_width}}" for c in desc.columns)
    )
    lines.append("  " + "-" * 12 + "-" * col_width * len(desc.columns))

    for stat in desc.index:
        cells = []
        for col in desc.columns:
            val = desc.loc[stat, col]
            cell = f"{val:<{col_width}.2f}" if not pd.isna(val) else f"{'NaN':<{col_width}}"
            cells.append(cell)
        lines.append(f"  {stat:<12} {''.join(cells)}")
    lines.append("")


def _top15_section(df: pd.DataFrame, lines: list[str]) -> None:
    """Top-15 frequency tables for key categorical columns."""
    for col in ["town", "flat_type", "flat_model", "storey_range"]:
        _heading(lines, f"Top-15 Values — {col}")
        counts = df[col].value_counts().head(15)
        label_width = max(len(str(k)) for k in counts.index) + 2

        lines.extend([
            f"  {'Value':<{label_width}} {'Count':>10} {'%':>8}",
            f"  {'-' * label_width} {'-' * 10} {'-' * 8}",
        ])
        for value, count in counts.items():
            lines.append(
                f"  {str(value):<{label_width}} {count:>10,} "
                f"{count / len(df) * 100:>7.2f}%"
            )
        lines.append("")


def _remaining_sections(df: pd.DataFrame, lines: list[str]) -> None:
    """Source breakdown, missing values, duplicates, outliers, yearly counts."""
    null_counts = df.isnull().sum()
    null_pcts = null_counts / len(df) * 100

    # Records per source file
    _heading(lines, "Records per Source File")
    for label, count in df["source"].value_counts().items():
        lines.append(f"  {label:<35} {count:>10,}  ({count / len(df) * 100:5.2f}%)")
    lines.append("")

    # Missing value summary
    _heading(lines, "Missing Value Summary")
    for col in null_counts[null_counts > 0].index:
        lines.append(
            f"  {col:<25} {null_counts[col]:>10,} missing "
            f"({null_pcts[col]:.2f}%)"
        )
    lines.append("")

    # Duplicate analysis
    _heading(lines, "Duplicate Analysis")
    exact_dupes = df.duplicated(
        subset=[c for c in df.columns if c != "source"]
    ).sum()
    lines.append(
        f"  Exact duplicates (ignoring source) : {exact_dupes:>10,}  "
        f"({exact_dupes / len(df) * 100:.2f}%)"
    )
    key_dupes = df.duplicated(
        subset=["month", "town", "block", "flat_type", "storey_range"]
    ).sum()
    lines.append(
        f"  Duplicates on key columns          : {key_dupes:>10,}  "
        f"({key_dupes / len(df) * 100:.2f}%)"
    )
    lines.append("")

    # IQR-based outlier detection on resale price
    _heading(lines, "Potential Outliers — Resale Price (IQR)")
    q1, q3 = df["resale_price"].quantile([0.25, 0.75])
    iqr = q3 - q1
    lower_fence = q1 - 1.5 * iqr
    upper_fence = q3 + 1.5 * iqr
    n_outliers = ((df["resale_price"] < lower_fence) | (df["resale_price"] > upper_fence)).sum()
    lines.extend([
        f"  Q1           : ${q1:>10,.2f}",
        f"  Q3           : ${q3:>10,.2f}",
        f"  IQR          : ${iqr:>10,.2f}",
        f"  Lower fence  : ${lower_fence:>10,.2f}",
        f"  Upper fence  : ${upper_fence:>10,.2f}",
        f"  Outliers     : {n_outliers:>10,}  ({n_outliers / len(df) * 100:.2f}%)",
        "",
    ])

    # Year-over-year record counts
    _heading(lines, "Records per Year")
    for year, count in df["month"].dt.year.value_counts().sort_index().items():
        lines.append(f"  {int(year):<6}  {count:>10,}")
    lines.append("")


# ── Generate the report ─────────────────────────────────────────────────────

lines: list[str] = []
_overview_section(master, lines)
_column_profile_section(master, lines)
_numeric_stats_section(master, lines)
_top15_section(master, lines)
_remaining_sections(master, lines)
_heading(lines, "End of Profiling Report")

report = "\n".join(lines)

# Save to disk
CFG.report_dir.mkdir(parents=True, exist_ok=True)
CFG.profile_path.write_text(report, encoding="utf-8")
print(f"Profiling report -> {CFG.profile_path}")

Profiling report -> /Users/john.yap/Desktop/DEA/profiling_outputs/profiling_summary.txt


In [7]:
# ── Report helpers ──────────────────────────────────────────────────────────

SEPARATOR = "-" * 65


@dataclass(frozen=True, slots=True)
class DataFrameStats:
    """Pre-computed statistics shared across all report sections.
    """

    n_rows: int
    n_cols: int
    memory_mb: float
    null_counts: pd.Series
    null_pcts: pd.Series
    numeric_cols: list[str]

    @classmethod
    def from_dataframe(cls, df: pd.DataFrame) -> DataFrameStats:
        n = len(df)
        nulls = df.isnull().sum()
        return cls(
            n_rows=n,
            n_cols=len(df.columns),
            memory_mb=df.memory_usage(deep=True).sum() / 1024**2,
            null_counts=nulls,
            null_pcts=(nulls / n * 100) if n else (nulls * 0.0),
            numeric_cols=df.select_dtypes(include=[np.number]).columns.tolist(),
        )


def _heading(lines: list[str], title: str) -> None:
    """Append a section divider with a title."""
    lines.extend([SEPARATOR, f"  {title}", SEPARATOR])


def _pct(part: int, total: int) -> float:
    """Safe percentage (returns 0.0 when *total* is zero)."""
    return part / total * 100 if total else 0.0


# ── Report sections ─────────────────────────────────────────────────────────

def _overview_section(
    df: pd.DataFrame, stats: DataFrameStats, lines: list[str]
) -> None:
    """Dataset-level summary: shape, memory, date range, cardinalities."""
    _heading(lines, "Dataset Overview")
    lines.extend([
        f"  Total rows              : {stats.n_rows:,}",
        f"  Total columns           : {stats.n_cols}",
        f"  Memory (approx)         : {stats.memory_mb:.2f} MB",
        f"  Date range              : {df['month'].min():%Y-%m}  to  {df['month'].max():%Y-%m}",
        f"  Distinct towns          : {df['town'].nunique()}",
        f"  Distinct flat types     : {df['flat_type'].nunique()}",
        f"  Distinct flat models    : {df['flat_model'].nunique()}",
        f"  Distinct storey ranges  : {df['storey_range'].nunique()}",
        "",
    ])


def _column_profile_section(
    df: pd.DataFrame, stats: DataFrameStats, lines: list[str]
) -> None:
    """Per-column summary: data type, non-null count, null %, unique values."""
    _heading(lines, "Column Profiles")
    header = (
        f"  {'Column':<25} {'Type':<16} {'Non-Null':>10} {'Null%':>7} {'Unique':>8}"
    )
    lines.extend([header, "  " + "-" * (len(header) - 2)])

    for col in df.columns:
        nulls = stats.null_counts[col]
        lines.append(
            f"  {col:<25} {str(df[col].dtype):<16} "
            f"{stats.n_rows - nulls:>10,} {_pct(nulls, stats.n_rows):>6.1f}% "
            f"{df[col].nunique():>8,}"
        )
    lines.append("")


def _numeric_stats_section(
    df: pd.DataFrame, stats: DataFrameStats, lines: list[str]
) -> None:
    """Descriptive statistics for numeric columns."""
    if not stats.numeric_cols:
        return

    _heading(lines, "Numeric Descriptive Statistics")
    desc = df[stats.numeric_cols].describe(
        percentiles=[0.01, 0.05, 0.25, 0.5, 0.75, 0.95, 0.99]
    )
    col_width = max(len(c) for c in desc.columns) + 2

    lines.append(
        f"  {'Statistic':<12} "
        + "".join(f"{c:<{col_width}}" for c in desc.columns)
    )
    lines.append("  " + "-" * (12 + col_width * len(desc.columns)))

    for stat in desc.index:
        cells = "".join(
            f"{desc.loc[stat, c]:<{col_width}.2f}"
            if not pd.isna(desc.loc[stat, c])
            else f"{'NaN':<{col_width}}"
            for c in desc.columns
        )
        lines.append(f"  {stat:<12} {cells}")
    lines.append("")


def _top_values_section(
    df: pd.DataFrame,
    stats: DataFrameStats,
    lines: list[str],
    columns: list[str] | None = None,
    n: int = 15,
) -> None:
    """Top-*n* frequency tables for categorical columns."""
    if columns is None:
        columns = ["town", "flat_type", "flat_model", "storey_range"]

    for col in columns:
        _heading(lines, f"Top-{n} Values -- {col}")
        counts = df[col].value_counts().head(n)
        label_width = max(len(str(k)) for k in counts.index) + 2

        lines.extend([
            f"  {'Value':<{label_width}} {'Count':>10} {'%':>8}",
            f"  {'-' * label_width} {'-' * 10} {'-' * 8}",
        ])
        for value, count in counts.items():
            lines.append(
                f"  {str(value):<{label_width}} {count:>10,} "
                f"{_pct(count, stats.n_rows):>7.2f}%"
            )
        lines.append("")


def _source_breakdown_section(
    df: pd.DataFrame, stats: DataFrameStats, lines: list[str]
) -> None:
    """Record counts per source file."""
    _heading(lines, "Records per Source File")
    for label, count in df["source"].value_counts().items():
        lines.append(
            f"  {label:<35} {count:>10,}  "
            f"({_pct(count, stats.n_rows):5.2f}%)"
        )
    lines.append("")


def _missing_values_section(stats: DataFrameStats, lines: list[str]) -> None:
    """Columns with missing values."""
    _heading(lines, "Missing Value Summary")
    has_nulls = stats.null_counts[stats.null_counts > 0]
    if has_nulls.empty:
        lines.append("  No missing values.")
    else:
        for col in has_nulls.index:
            lines.append(
                f"  {col:<25} {stats.null_counts[col]:>10,} missing "
                f"({stats.null_pcts[col]:.2f}%)"
            )
    lines.append("")


def _duplicate_section(
    df: pd.DataFrame, stats: DataFrameStats, lines: list[str]
) -> None:
    """Exact and key-column duplicate counts."""
    _heading(lines, "Duplicate Analysis")

    non_source_cols = [c for c in df.columns if c != "source"]
    exact_dupes = df.duplicated(subset=non_source_cols).sum()
    lines.append(
        f"  Exact duplicates (ignoring source) : {exact_dupes:>10,}  "
        f"({_pct(exact_dupes, stats.n_rows):.2f}%)"
    )

    key_cols = ["month", "town", "block", "flat_type", "storey_range"]
    key_dupes = df.duplicated(subset=key_cols).sum()
    lines.append(
        f"  Duplicates on key columns          : {key_dupes:>10,}  "
        f"({_pct(key_dupes, stats.n_rows):.2f}%)"
    )
    lines.append("")


def _outlier_section(
    df: pd.DataFrame, stats: DataFrameStats, lines: list[str]
) -> None:
    """IQR-based outlier detection on resale_price."""
    _heading(lines, "Potential Outliers -- Resale Price (IQR)")
    q1, q3 = df["resale_price"].quantile([0.25, 0.75])
    iqr = q3 - q1
    lower, upper = q1 - 1.5 * iqr, q3 + 1.5 * iqr
    n_outliers = (
        (df["resale_price"] < lower) | (df["resale_price"] > upper)
    ).sum()

    lines.extend([
        f"  Q1           : ${q1:>10,.2f}",
        f"  Q3           : ${q3:>10,.2f}",
        f"  IQR          : ${iqr:>10,.2f}",
        f"  Lower fence  : ${lower:>10,.2f}",
        f"  Upper fence  : ${upper:>10,.2f}",
        f"  Outliers     : {n_outliers:>10,}  "
        f"({_pct(n_outliers, stats.n_rows):.2f}%)",
        "",
    ])


def _yearly_counts_section(df: pd.DataFrame, lines: list[str]) -> None:
    """Record counts per calendar year."""
    _heading(lines, "Records per Year")
    for year, count in df["month"].dt.year.value_counts().sort_index().items():
        lines.append(f"  {int(year):<6}  {count:>10,}")
    lines.append("")


# ── Orchestrator ────────────────────────────────────────────────────────────

def generate_profiling_report(df: pd.DataFrame) -> str:
    """Build and save a plain-text profiling report for *df*.

    Returns
    -------
    str
        The full report text.
    """
    stats = DataFrameStats.from_dataframe(df)
    lines: list[str] = []

    _overview_section(df, stats, lines)
    _column_profile_section(df, stats, lines)
    _numeric_stats_section(df, stats, lines)
    _top_values_section(df, stats, lines)
    _source_breakdown_section(df, stats, lines)
    _missing_values_section(stats, lines)
    _duplicate_section(df, stats, lines)
    _outlier_section(df, stats, lines)
    _yearly_counts_section(df, lines)
    _heading(lines, "End of Profiling Report")

    report = "\n".join(lines)

    CFG.ensure_directories()
    CFG.profile_path.write_text(report, encoding="utf-8")
    print(f"Profiling report -> {CFG.profile_path}")

    return report


report = generate_profiling_report(master)

Profiling report -> /Users/john.yap/Desktop/DEA/profiling_outputs/profiling_summary.txt


## 3. Rules

In [ ]:
# ── Validation rule generation ───────────────────────────────────────────────

# Fields to validate and their expected logical types
VALIDATION_FIELDS: dict[str, str] = {
    "month": "date",
    "town": "categorical",
    "flat_type": "categorical",
    "flat_model": "categorical",
    "storey_range": "categorical",
}


def _infer_date_rule(series: pd.Series) -> dict:
    """Derive validation constraints for a date/period column."""
    non_null = series.dropna()
    return {
        "type": "date",
        "min": str(non_null.min().strftime("%Y-%m")),
        "max": str(non_null.max().strftime("%Y-%m")),
        "n_unique": int(non_null.nunique()),
        "null_count": int(series.isna().sum()),
        "null_allowed": series.isna().any(),
    }


def _infer_categorical_rule(series: pd.Series) -> dict:
    """Derive validation constraints for a categorical column."""
    non_null = series.dropna()
    value_counts = non_null.value_counts()
    return {
        "type": "categorical",
        "valid_values": sorted(non_null.unique().tolist()),
        "n_unique": int(non_null.nunique()),
        "most_frequent": str(value_counts.idxmax()),
        "least_frequent": str(value_counts.idxmin()),
        "null_count": int(series.isna().sum()),
        "null_allowed": series.isna().any(),
    }


def _sort_storey_ranges(values: list[str]) -> list[str]:
    """Sort storey-range strings by their leading numeric value."""
    def _leading_ints(text: str) -> list[int]:
        nums = re.findall(r"\d+", text)
        return [int(n) for n in nums] if nums else [0]
    return sorted(values, key=_leading_ints)


def _infer_storey_range_rule(series: pd.Series) -> dict:
    """Derive validation constraints for the storey_range column.

    Extends the standard categorical rule with a regex pattern
    detected from the data and numerically sorted valid values.
    """
    rule = _infer_categorical_rule(series)
    rule["valid_values"] = _sort_storey_ranges(rule["valid_values"])

    sample = series.dropna().unique()
    if all(re.fullmatch(r"\d{2}\s+TO\s+\d{2}", str(v)) for v in sample):
        rule["detected_pattern"] = r"^\d{2}\s+TO\s+\d{2}$"

    return rule


# Dispatch table mapping logical type to its inference function
_RULE_BUILDERS: dict[str, callable] = {
    "date": _infer_date_rule,
    "categorical": _infer_categorical_rule,
}

# Column-specific overrides (takes precedence over the default builder)
_COLUMN_OVERRIDES: dict[str, callable] = {
    "storey_range": _infer_storey_range_rule,
}


def build_validation_rules(
    df: pd.DataFrame,
    fields: dict[str, str] | None = None,
) -> dict:
    """Generate validation rules from the statistical properties of *df*.

    Parameters
    ----------
    df : pd.DataFrame
        The master dataset.
    fields : dict[str, str], optional
        Mapping of column name to logical type.  Defaults to
        ``VALIDATION_FIELDS``.

    Returns
    -------
    dict
        Column name -> rule dictionary, ready for JSON serialisation.
    """
    if fields is None:
        fields = VALIDATION_FIELDS

    rules: dict[str, dict] = {}
    for col, logical_type in fields.items():
        if col not in df.columns:
            raise KeyError(f"Column '{col}' not found in DataFrame.")

        builder = _COLUMN_OVERRIDES.get(col, _RULE_BUILDERS[logical_type])
        rules[col] = builder(df[col])

    return rules


def save_validation_rules(df: pd.DataFrame) -> dict:
    """Build validation rules, persist to JSON, and print a summary.

    Returns
    -------
    dict
        The generated rules.
    """
    rules = build_validation_rules(df)

    CFG.ensure_directories()
    CFG.rules_path.write_text(
        json.dumps(rules, indent=2, default=str), encoding="utf-8"
    )

    print(f"{len(rules)} field rules -> {CFG.rules_path}")
    for col, meta in rules.items():
        n_unique = meta.get("n_unique", "n/a")
        nulls = meta.get("null_count", 0)
        print(f"  {col:<20} type={meta['type']:<15} unique={n_unique:<6} nulls={nulls}")

    return rules


rules = save_validation_rules(master)

5 field rules -> /Users/john.yap/Desktop/DEA/profiling_outputs/validation_rules.json
  month                type=date            unique=60     nulls=0
  town                 type=categorical     unique=26     nulls=0
  flat_type            type=categorical     unique=7      nulls=0
  flat_model           type=categorical     unique=20     nulls=0
  storey_range         type=categorical     unique=25     nulls=0


In [10]:
# ── Validation engine ────────────────────────────────────────────────────────

@dataclass
class ValidationResult:
    """Outcome of validating a single column against its rule.

    Attributes
    ----------
    column : str
        Column name that was validated.
    passed : bool
        True if every row satisfied the rule.
    n_violations : int
        Number of rows that violated the rule.
    violation_pct : float
        Percentage of total rows that violated the rule.
    sample_violations : list
        Up to 10 example values that failed validation.
    details : str
        Human-readable explanation of what was checked.
    """

    column: str
    passed: bool
    n_violations: int
    violation_pct: float
    sample_violations: list
    details: str


def _validate_nulls(series: pd.Series, rule: dict) -> pd.Series:
    """Return a boolean mask where True = row violates the null constraint."""
    if rule.get("null_allowed", False):
        return pd.Series(False, index=series.index)
    return series.isna()


def _validate_date_column(series: pd.Series, rule: dict) -> ValidationResult:
    """Check that all date values fall within [min, max]."""
    null_mask = _validate_nulls(series, rule)
    non_null = series.dropna()

    out_of_range = (
        (non_null < pd.Timestamp(rule["min"]))
        | (non_null > pd.Timestamp(rule["max"]) + pd.offsets.MonthEnd(0))
    )

    violation_mask = null_mask.copy()
    violation_mask.loc[out_of_range.index] = violation_mask.loc[out_of_range.index] | out_of_range

    n_bad = int(violation_mask.sum())
    bad_values = series[violation_mask].dropna().unique().tolist()[:10]

    return ValidationResult(
        column=series.name,
        passed=(n_bad == 0),
        n_violations=n_bad,
        violation_pct=_pct(n_bad, len(series)),
        sample_violations=[str(v) for v in bad_values],
        details=f"Date must be in [{rule['min']}, {rule['max']}], "
                f"null_allowed={rule.get('null_allowed', False)}",
    )


def _validate_categorical_column(series: pd.Series, rule: dict) -> ValidationResult:
    """Check that all values belong to the set of valid values."""
    null_mask = _validate_nulls(series, rule)
    non_null = series.dropna()

    valid_set = set(rule["valid_values"])
    invalid_mask = ~non_null.isin(valid_set)

    violation_mask = null_mask.copy()
    violation_mask.loc[invalid_mask.index] = (
        violation_mask.loc[invalid_mask.index] | invalid_mask
    )

    if "detected_pattern" in rule:
        pattern = re.compile(rule["detected_pattern"])
        pattern_fail = non_null.apply(lambda v: not pattern.match(str(v)))
        violation_mask.loc[pattern_fail.index] = (
            violation_mask.loc[pattern_fail.index] | pattern_fail
        )

    n_bad = int(violation_mask.sum())
    bad_values = series[violation_mask].dropna().unique().tolist()[:10]

    return ValidationResult(
        column=series.name,
        passed=(n_bad == 0),
        n_violations=n_bad,
        violation_pct=_pct(n_bad, len(series)),
        sample_violations=bad_values,
        details=f"Must be one of {len(valid_set)} valid values, "
                f"null_allowed={rule.get('null_allowed', False)}",
    )


_VALIDATORS: dict[str, callable] = {
    "date": _validate_date_column,
    "categorical": _validate_categorical_column,
}


def validate_dataframe(
    df: pd.DataFrame, rules: dict
) -> list[ValidationResult]:
    """Run all validation rules against *df* and return results.

    Parameters
    ----------
    df : pd.DataFrame
        The dataset to validate.
    rules : dict
        Rules as produced by ``build_validation_rules``.

    Returns
    -------
    list[ValidationResult]
        One result per validated column.
    """
    results: list[ValidationResult] = []

    for col, rule in rules.items():
        if col not in df.columns:
            results.append(ValidationResult(
                column=col,
                passed=False,
                n_violations=len(df),
                violation_pct=100.0,
                sample_violations=[],
                details=f"Column '{col}' is missing from the DataFrame.",
            ))
            continue

        validator = _VALIDATORS.get(rule["type"])
        if validator is None:
            raise ValueError(
                f"No validator registered for type '{rule['type']}' "
                f"(column '{col}')."
            )
        results.append(validator(df[col], rule))

    return results


def print_validation_report(results: list[ValidationResult]) -> None:
    """Print a summary table of validation results."""
    print(f"\n{SEPARATOR}")
    print("  Validation Report")
    print(SEPARATOR)

    all_passed = True
    for r in results:
        status = "PASS" if r.passed else "FAIL"
        if not r.passed:
            all_passed = False
        print(
            f"  [{status}]  {r.column:<20}  "
            f"violations={r.n_violations:,} ({r.violation_pct:.2f}%)"
        )
        if r.sample_violations:
            print(f"         examples: {r.sample_violations}")

    print(SEPARATOR)
    if all_passed:
        print("  All columns passed validation.")
    else:
        failed = [r.column for r in results if not r.passed]
        print(f"  {len(failed)} column(s) failed: {', '.join(failed)}")
    print()


# ── Run validation on the master dataset ─────────────────────────────────────

results = validate_dataframe(master, rules)
print_validation_report(results)


# ── Test with deliberately bad data to confirm rules catch violations ────────

def test_validation_catches_violations(df: pd.DataFrame, rules: dict) -> None:
    """Inject known-bad values and verify the validator flags them."""
    bad_data = df.head(5).copy()

    bad_data.loc[bad_data.index[0], "month"] = pd.Timestamp("1999-01-01")
    bad_data.loc[bad_data.index[1], "town"] = "FAKE_TOWN"
    bad_data.loc[bad_data.index[2], "flat_type"] = "99 ROOM"
    bad_data.loc[bad_data.index[3], "flat_model"] = "Nonexistent Model"
    bad_data.loc[bad_data.index[4], "storey_range"] = "XX TO YY"

    bad_results = validate_dataframe(bad_data, rules)

    print(f"\n{SEPARATOR}")
    print("  Validation Test (injected bad data)")
    print(SEPARATOR)

    expected_failures = {"month", "town", "flat_type", "flat_model", "storey_range"}
    actual_failures = {r.column for r in bad_results if not r.passed}

    for r in bad_results:
        status = "PASS" if r.passed else "FAIL"
        print(f"  [{status}]  {r.column:<20}  violations={r.n_violations}")
        if r.sample_violations:
            print(f"         caught: {r.sample_violations}")

    print(SEPARATOR)

    missed = expected_failures - actual_failures
    if not missed:
        print("  Test passed: all injected violations were detected.")
    else:
        print(f"  Test failed: missed violations in: {', '.join(missed)}")
    print()


test_validation_catches_violations(master, rules)


-----------------------------------------------------------------
  Validation Report
-----------------------------------------------------------------
  [PASS]  month                 violations=0 (0.00%)
  [PASS]  town                  violations=0 (0.00%)
  [PASS]  flat_type             violations=0 (0.00%)
  [PASS]  flat_model            violations=0 (0.00%)
  [PASS]  storey_range          violations=0 (0.00%)
-----------------------------------------------------------------
  All columns passed validation.


-----------------------------------------------------------------
  Validation Test (injected bad data)
-----------------------------------------------------------------
  [FAIL]  month                 violations=1
         caught: ['1999-01-01 00:00:00']
  [FAIL]  town                  violations=1
         caught: ['FAKE_TOWN']
  [FAIL]  flat_type             violations=1
         caught: ['99 ROOM']
  [FAIL]  flat_model            violations=1
         caught: ['Nonexistent

## 4. Lease

In [11]:
# ── Lease computation ────────────────────────────────────────────────────────

HDB_LEASE_YEARS: int = 99


def _compute_remaining_lease_columns(df: pd.DataFrame) -> pd.DataFrame:
    """Add remaining lease columns to *df* using vectorised arithmetic.

    For each row the lease runs from Jan 1 of ``lease_commence_date`` to
    Jan 1 of ``lease_commence_date + 99``.  The remaining time is measured
    from today's date.

    Columns added
    -------------
    remaining_lease_years : Int64
        Whole years remaining (nullable integer).
    remaining_lease_months : Int64
        Additional whole months beyond the year count (0-11, nullable).
    remaining_lease_computed : str or pd.NA
        Human-readable string, e.g. ``"61 years 04 months"``.

    Parameters
    ----------
    df : pd.DataFrame
        Must contain a ``lease_commence_date`` column with numeric years.

    Returns
    -------
    pd.DataFrame
        The input DataFrame with the three new columns appended.
    """
    commence = df["lease_commence_date"]
    valid = commence.notna()

    lease_end_year = commence[valid].astype(int) + HDB_LEASE_YEARS

    total_months = (
        (lease_end_year - TODAY.year) * MONTHS_PER_YEAR
        + (1 - TODAY.month)
    )

    # Leases that have already expired
    total_months = total_months.clip(lower=0)

    years = total_months // MONTHS_PER_YEAR
    months = total_months % MONTHS_PER_YEAR

    df["remaining_lease_years"] = pd.array(
        [pd.NA] * len(df), dtype="Int64"
    )
    df["remaining_lease_months"] = pd.array(
        [pd.NA] * len(df), dtype="Int64"
    )

    df.loc[valid, "remaining_lease_years"] = years.astype("Int64")
    df.loc[valid, "remaining_lease_months"] = months.astype("Int64")

    formatted = (
        years.astype(int).astype(str) + " years "
        + months.astype(int).map("{:02d}".format) + " months"
    )
    df["remaining_lease_computed"] = pd.NA
    df.loc[valid, "remaining_lease_computed"] = formatted

    return df


def _print_lease_summary(df: pd.DataFrame) -> None:
    """Print a brief summary of the computed lease columns."""
    has_lease = df["remaining_lease_years"].notna()
    n_computed = int(has_lease.sum())
    n_expired = int((df.loc[has_lease, "remaining_lease_years"] == 0).sum())

    print(f"  Computed for {n_computed:,} records")
    print(f"  Expired (0 remaining): {n_expired:,}")
    print("  Sample:")

    sample = (
        df.loc[has_lease, [
            "lease_commence_date",
            "remaining_lease_years",
            "remaining_lease_months",
        ]]
        .head(10)
    )
    for _, row in sample.iterrows():
        print(
            f"    Lease {int(row['lease_commence_date'])} -> "
            f"{int(row['remaining_lease_years'])}y "
            f"{int(row['remaining_lease_months']):02d}m"
        )


# ── Apply ────────────────────────────────────────────────────────────────────

master = _compute_remaining_lease_columns(master)
_print_lease_summary(master)

  Computed for 92,544 records
  Expired (0 remaining): 0
  Sample:
    Lease 1979 -> 51y 06m
    Lease 2000 -> 72y 06m
    Lease 2001 -> 73y 06m
    Lease 1999 -> 71y 06m
    Lease 2000 -> 72y 06m
    Lease 2000 -> 72y 06m
    Lease 2003 -> 75y 06m
    Lease 1999 -> 71y 06m
    Lease 1999 -> 71y 06m
    Lease 2000 -> 72y 06m


## 5. Dedup

In [12]:
# ── Composite-key deduplication ──────────────────────────────────────────────

def deduplicate_by_key(
    df: pd.DataFrame,
    price_col: str = "resale_price",
) -> tuple[pd.DataFrame, pd.DataFrame]:
    """Remove duplicate rows that share the same composite key.

    The composite key is every column except *price_col*.  When multiple
    rows share a key, the row with the highest price is kept; the rest
    are returned separately as the "failed" set.

    Parameters
    ----------
    df : pd.DataFrame
        The dataset to deduplicate.
    price_col : str
        Column used to break ties (highest value wins).

    Returns
    -------
    tuple[pd.DataFrame, pd.DataFrame]
        (passed, failed) — the deduplicated dataset and the discarded rows.
    """
    key_columns = [c for c in df.columns if c != price_col]

    ranked = df.assign(
        _rank=df.groupby(key_columns)[price_col].rank(
            method="first", ascending=False
        )
    )

    passed = ranked.loc[ranked["_rank"] == 1].drop(columns="_rank")
    failed = ranked.loc[ranked["_rank"] > 1].drop(columns="_rank")

    return passed, failed


def _print_dedup_summary(
    before: int, passed: pd.DataFrame, failed: pd.DataFrame
) -> None:
    """Print a one-line-per-category deduplication summary."""
    print(f"  Before:   {before:,} rows")
    print(f"  Passed:   {len(passed):,} (highest price kept)")
    print(f"  Failed:   {len(failed):,} (lower-price duplicates removed)")


# ── Apply ────────────────────────────────────────────────────────────────────

before = len(master)
master, deduped_fail = deduplicate_by_key(master)
_print_dedup_summary(before, master, deduped_fail)

  Before:   92,544 rows
  Passed:   36,683 (highest price kept)
  Failed:   470 (lower-price duplicates removed)


## 6. Anomaly

In [13]:
# ── Anomaly detection ────────────────────────────────────────────────────────

IQR_MULTIPLIER: float = 1.5
Z_SCORE_THRESHOLD: float = 3.0


def _flag_global_iqr(prices: pd.Series) -> pd.Series:
    """Flag prices outside 1.5x IQR from the dataset quartiles."""
    q1, q3 = prices.quantile([0.25, 0.75])
    iqr = q3 - q1
    lower = q1 - IQR_MULTIPLIER * iqr
    upper = q3 + IQR_MULTIPLIER * iqr
    return (prices < lower) | (prices > upper)


def _flag_group_zscore(
    df: pd.DataFrame,
    value_col: str,
    group_cols: list[str],
) -> pd.Series:
    """Flag rows whose *value_col* z-score exceeds the threshold
    within each group defined by *group_cols*."""
    z = df.groupby(group_cols)[value_col].transform(
        lambda x: (x - x.mean()) / x.std() if x.std() > 0 else 0.0
    )
    return z.abs() > Z_SCORE_THRESHOLD


@dataclass(frozen=True, slots=True)
class HeuristicResult:
    """Outcome of a single anomaly heuristic."""

    name: str
    description: str
    mask: pd.Series


def _build_heuristics(df: pd.DataFrame) -> list[HeuristicResult]:
    """Run all anomaly heuristics and return their results."""
    price_per_sqm = df["resale_price"] / df["floor_area_sqm"]

    return [
        HeuristicResult(
            name="global_iqr",
            description="Price outside 1.5x IQR from dataset quartiles",
            mask=_flag_global_iqr(df["resale_price"]),
        ),
        HeuristicResult(
            name="psqm_town_flattype_z3",
            description=(
                f"Price/sqm z-score > {Z_SCORE_THRESHOLD} "
                "within same town + flat_type"
            ),
            mask=_flag_group_zscore(
                df.assign(_psqm=price_per_sqm),
                "_psqm",
                ["town", "flat_type"],
            ),
        ),
        HeuristicResult(
            name="psqm_flatmodel_z3",
            description=(
                f"Price/sqm z-score > {Z_SCORE_THRESHOLD} "
                "within same flat_model"
            ),
            mask=_flag_group_zscore(
                df.assign(_psqm=price_per_sqm),
                "_psqm",
                ["flat_model"],
            ),
        ),
    ]


def detect_anomalies(
    df: pd.DataFrame,
) -> tuple[pd.DataFrame, pd.DataFrame, dict]:
    """Split *df* into clean and anomalous subsets.

    A row is anomalous if any heuristic flags it.

    Parameters
    ----------
    df : pd.DataFrame
        The dataset to scan.

    Returns
    -------
    tuple[pd.DataFrame, pd.DataFrame, dict]
        (clean, anomalous, summary) where summary is a JSON-serialisable
        dict describing what was found.
    """
    heuristics = _build_heuristics(df)

    combined_mask = pd.Series(False, index=df.index)
    heuristic_details: dict[str, dict] = {}

    for h in heuristics:
        combined_mask = combined_mask | h.mask
        heuristic_details[h.name] = {
            "desc": h.description,
            "flagged": int(h.mask.sum()),
        }

    clean = df.loc[~combined_mask].copy()
    anomalous = df.loc[combined_mask].copy()

    summary = {
        "total_rows": len(df),
        "anomalous_rows": len(anomalous),
        "anomaly_pct": round(_pct(len(anomalous), len(df)), 2),
        "clean_rows": len(clean),
        "heuristics": heuristic_details,
        "assumptions": [
            "99-year HDB lease assumed; remaining lease computed "
            "from lease_commence_date.",
            f"z-score threshold of {Z_SCORE_THRESHOLD} captures approximately "
            "0.3% tail events under normality.",
            "Peer groups use town + flat_type and flat_model independently.",
            "Price per sqm normalises for unit size variation.",
        ],
    }

    return clean, anomalous, summary


def _print_anomaly_report(summary: dict) -> None:
    """Print a formatted anomaly detection report."""
    total = summary["total_rows"]

    print(f"\n{SEPARATOR}")
    print("  Anomaly Detection Results")
    print(SEPARATOR)
    print(f"  Total records checked : {total:,}")
    print(f"  Anomalous records     : {summary['anomalous_rows']:,} "
          f"({summary['anomaly_pct']}%)")
    print(f"  Clean records         : {summary['clean_rows']:,}")

    print(f"\n  Breakdown by heuristic:")
    print(f"  {'-' * 50}")
    for name, detail in summary["heuristics"].items():
        pct = _pct(detail["flagged"], total)
        print(f"    {name:<30} {detail['flagged']:>6,} rows ({pct:.2f}%)")
        print(f"      {detail['desc']}")

    print(SEPARATOR)


# ── Apply ────────────────────────────────────────────────────────────────────

clean_data, anomalous_data, anomaly_summary = detect_anomalies(master)

CFG.ensure_directories()
(CFG.report_dir / "anomaly_summary.json").write_text(
    json.dumps(anomaly_summary, indent=2), encoding="utf-8"
)

_print_anomaly_report(anomaly_summary)


-----------------------------------------------------------------
  Anomaly Detection Results
-----------------------------------------------------------------
  Total records checked : 36,683
  Anomalous records     : 2,166 (5.9%)
  Clean records         : 34,517

  Breakdown by heuristic:
  --------------------------------------------------
    global_iqr                      1,751 rows (4.77%)
      Price outside 1.5x IQR from dataset quartiles
    psqm_town_flattype_z3             271 rows (0.74%)
      Price/sqm z-score > 3.0 within same town + flat_type
    psqm_flatmodel_z3                 465 rows (1.27%)
      Price/sqm z-score > 3.0 within same flat_model
-----------------------------------------------------------------


In [14]:
# ── Additional validation checks ────────────────────────────────────────────

from collections import Counter

from sklearn.ensemble import IsolationForest

# Benford's Law expected leading-digit frequencies (%)
_BENFORD_EXPECTED: dict[int, float] = {
    1: 30.1, 2: 17.6, 3: 12.5, 4: 9.7, 5: 7.9,
    6: 6.7,  7: 5.8,  8: 5.1,  9: 4.6,
}

# Isolation Forest defaults
_ISO_CONTAMINATION: float = 0.02
_ISO_ESTIMATORS: int = 200
_ISO_SEED: int = 42
_ISO_FEATURES: list[str] = [
    "resale_price", "floor_area_sqm", "lease_commence_date", "_price_per_sqm",
]

# Year-over-year threshold
_YOY_THRESHOLD: float = 0.20


# ── Benford's Law ───────────────────────────────────────────────────────────

def check_benfords_law(prices: pd.Series) -> tuple[pd.DataFrame, float]:
    """Compare leading-digit distribution of *prices* against Benford's Law.

    Parameters
    ----------
    prices : pd.Series
        Numeric series (e.g. resale prices).

    Returns
    -------
    tuple[pd.DataFrame, float]
        A DataFrame with columns (digit, expected_pct, actual_pct, deviation)
        and the maximum absolute deviation across all digits.
    """
    leading = (
        prices.dropna()
        .astype(int)
        .astype(str)
        .str[0]
        .astype(int)
    )
    counts = Counter(leading)
    total = sum(counts.values())

    rows = []
    for digit in range(1, 10):
        actual = counts.get(digit, 0) / total * 100
        expected = _BENFORD_EXPECTED[digit]
        rows.append({
            "digit": digit,
            "expected_pct": expected,
            "actual_pct": round(actual, 1),
            "deviation": round(abs(actual - expected), 1),
        })

    result = pd.DataFrame(rows)
    return result, float(result["deviation"].max())


# ── Isolation Forest ────────────────────────────────────────────────────────

def check_isolation_forest(df: pd.DataFrame) -> tuple[int, int]:
    """Run an Isolation Forest over numeric features to flag multivariate
    outliers.

    Parameters
    ----------
    df : pd.DataFrame
        Must contain the columns listed in ``_ISO_FEATURES`` (or
        ``_price_per_sqm`` will be computed on the fly).

    Returns
    -------
    tuple[int, int]
        (n_flagged, n_total).
    """
    work = df.copy()
    if "_price_per_sqm" not in work.columns:
        work["_price_per_sqm"] = work["resale_price"] / work["floor_area_sqm"]

    features = work[_ISO_FEATURES].fillna(work[_ISO_FEATURES].median())

    labels = IsolationForest(
        contamination=_ISO_CONTAMINATION,
        random_state=_ISO_SEED,
        n_estimators=_ISO_ESTIMATORS,
    ).fit_predict(features)

    return int((labels == -1).sum()), len(work)


# ── Year-over-year price jumps ──────────────────────────────────────────────

def check_yoy_jumps(
    df: pd.DataFrame,
    threshold: float = _YOY_THRESHOLD,
) -> pd.DataFrame:
    """Identify (town, flat_type, year) groups where the median resale price
    changed by more than *threshold* relative to the previous year.

    Returns
    -------
    pd.DataFrame
        Flagged rows with columns: town, flat_type, year, median_price,
        prev_year_median, yoy_change.
    """
    yearly = (
        df.groupby([
            "town",
            "flat_type",
            df["month"].dt.year.rename("year"),
        ])["resale_price"]
        .median()
        .reset_index()
        .sort_values(["town", "flat_type", "year"])
    )

    yearly["prev_year_median"] = (
        yearly.groupby(["town", "flat_type"])["resale_price"].shift(1)
    )
    yearly["yoy_change"] = (
        (yearly["resale_price"] - yearly["prev_year_median"])
        / yearly["prev_year_median"]
    )

    yearly = yearly.rename(columns={"resale_price": "median_price"})

    return yearly.loc[yearly["yoy_change"].abs() > threshold].copy()


# ── Orchestrator ────────────────────────────────────────────────────────────

def run_additional_checks(df: pd.DataFrame) -> dict:
    """Run all supplementary validation checks and print results.

    Parameters
    ----------
    df : pd.DataFrame
        The master (or clean) dataset.

    Returns
    -------
    dict
        Summary dict suitable for JSON serialisation.
    """
    benford_df, max_dev = check_benfords_law(df["resale_price"])
    n_iso, n_total = check_isolation_forest(df)
    yoy_flagged = check_yoy_jumps(df)

    # -- Print results --

    print(f"\n{SEPARATOR}")
    print("  Additional Validation Checks")
    print(SEPARATOR)

    # Benford
    print("\n  1. Benford's Law -- Leading Digit Distribution")
    print(f"     {'Digit':<8} {'Expected':>10} {'Actual':>10} {'Deviation':>10}")
    print(f"     {'-' * 8} {'-' * 10} {'-' * 10} {'-' * 10}")
    for _, r in benford_df.iterrows():
        print(
            f"     {int(r['digit']):<8} {r['expected_pct']:>9.1f}% "
            f"{r['actual_pct']:>9.1f}% {r['deviation']:>8.1f}%"
        )
    print(f"\n     Max deviation: {max_dev:.1f}%")

    # Isolation Forest
    print(f"\n  2. Isolation Forest -- Multivariate Anomaly Detection")
    print(
        f"     Flagged: {n_iso:,} / {n_total:,} "
        f"({_pct(n_iso, n_total):.2f}%)"
    )
    print(f"     Features: {', '.join(_ISO_FEATURES)}")

    # YoY
    print(f"\n  3. Year-over-Year Price Jumps (>{_YOY_THRESHOLD:.0%} median change)")
    print(f"     Groups flagged: {len(yoy_flagged)}")
    if not yoy_flagged.empty:
        print(f"     {'Town':<20} {'Flat Type':<18} {'Year':>6} {'Change':>10}")
        print(f"     {'-' * 20} {'-' * 18} {'-' * 6} {'-' * 10}")
        for _, r in yoy_flagged.iterrows():
            print(
                f"     {r['town']:<20} {r['flat_type']:<18} "
                f"{int(r['year']):>6} {r['yoy_change']:>+9.1%}"
            )

    # Summary table
    benford_verdict = "Expected" if max_dev < 20 else "Review"
    iso_verdict = "Review" if n_iso > 0 else "Clean"
    yoy_verdict = "Stable" if yoy_flagged.empty else "Review"

    print(f"\n{SEPARATOR}")
    print("  Summary")
    print(SEPARATOR)
    print(f"  {'Check':<26} {'Result':<26} {'Verdict'}")
    print(f"  {'-' * 26} {'-' * 26} {'-' * 12}")
    print(f"  {'Benfords Law':<26} {'Max dev ' + f'{max_dev:.1f}%':<26} {benford_verdict}")
    print(f"  {'Isolation Forest':<26} {f'{n_iso:,} rows ({_pct(n_iso, n_total):.1f}%)':<26} {iso_verdict}")
    print(f"  {'YoY Price Jumps':<26} {f'{len(yoy_flagged)} group(s)':<26} {yoy_verdict}")
    print(SEPARATOR)

    return {
        "benfords_law": {
            "max_deviation_pct": max_dev,
            "verdict": benford_verdict,
        },
        "isolation_forest": {
            "flagged": n_iso,
            "total": n_total,
            "pct": round(_pct(n_iso, n_total), 2),
            "verdict": iso_verdict,
        },
        "yoy_jumps": {
            "groups_flagged": len(yoy_flagged),
            "threshold": _YOY_THRESHOLD,
            "verdict": yoy_verdict,
        },
    }


# ── Apply ────────────────────────────────────────────────────────────────────

additional_checks = run_additional_checks(master)


-----------------------------------------------------------------
  Additional Validation Checks
-----------------------------------------------------------------

  1. Benford's Law -- Leading Digit Distribution
     Digit      Expected     Actual  Deviation
     -------- ---------- ---------- ----------
     1             30.1%       0.1%     30.0%
     2             17.6%      11.0%      6.6%
     3             12.5%      35.3%     22.8%
     4              9.7%      29.1%     19.4%
     5              7.9%      12.0%      4.1%
     6              6.7%       6.7%      0.0%
     7              5.8%       3.4%      2.4%
     8              5.1%       1.8%      3.3%
     9              4.6%       0.7%      3.9%

     Max deviation: 30.0%

  2. Isolation Forest -- Multivariate Anomaly Detection
     Flagged: 734 / 36,683 (2.00%)
     Features: resale_price, floor_area_sqm, lease_commence_date, _price_per_sqm

  3. Year-over-Year Price Jumps (>20% median change)
     Groups flagged: 0



## 8. Identifier + Export

In [15]:
# ── Record ID generation ────────────────────────────────────────────────────

import hashlib


def _normalise_block(block: str) -> str:
    """Extract up to 3 leading digits from a block number, zero-padded.

    Examples: '123A' -> '123', '45' -> '045', 'N/A' -> '000'.
    """
    digits = re.sub(r"\D", "", str(block))[:3]
    return digits.zfill(3)


def generate_record_ids(df: pd.DataFrame) -> pd.Series:
    """Build a composite record ID for each row.

    Format: S{block_digits}{avg_prefix}{month}{town_initial}

    Components
    ----------
    block_digits : str
        First 3 digits of the block number, zero-padded.
    avg_prefix : str
        First 2 characters of the stringified group mean price
        (grouped by year-month, town, flat_type).
    month : str
        Two-digit transaction month.
    town_initial : str
        First character of the town name (uppercased).
    """
    block_digits = df["block"].apply(_normalise_block)

    year_month = df["month"].dt.strftime("%Y-%m")
    group_mean = (
        df.groupby([year_month, "town", "flat_type"])["resale_price"]
        .transform("mean")
        .astype(str)
        .str[:2]
    )

    month_str = df["month"].dt.strftime("%m")
    town_initial = df["town"].str[0].str.upper()

    return "S" + block_digits + group_mean + month_str + town_initial


def generate_sha256(record_id: pd.Series) -> pd.Series:
    """Compute a SHA-256 hex digest for each value in *record_id*."""
    return record_id.apply(lambda x: hashlib.sha256(x.encode()).hexdigest())


# ── ID-based deduplication ──────────────────────────────────────────────────

def deduplicate_by_record_id(
    df: pd.DataFrame,
) -> tuple[pd.DataFrame, pd.DataFrame]:
    """Assign record IDs, then deduplicate (highest price wins).

    Parameters
    ----------
    df : pd.DataFrame
        The clean dataset.

    Returns
    -------
    tuple[pd.DataFrame, pd.DataFrame]
        (kept, discarded) DataFrames, each with a ``rid`` column.
    """
    work = df.copy()
    work["rid"] = generate_record_ids(work)

    n_unique = work["rid"].nunique()
    print(f"  Record IDs: {n_unique:,} unique / {len(work):,} total")

    rank = work.groupby("rid")["resale_price"].rank(
        method="first", ascending=False
    )
    kept = work.loc[rank == 1].copy()
    discarded = work.loc[rank > 1].copy()

    print(f"  Dedup: {len(kept):,} kept, {len(discarded):,} discarded")

    kept["id_hash"] = generate_sha256(kept["rid"])
    n_ids = kept["rid"].nunique()
    n_hashes = kept["id_hash"].nunique()
    print(f"  SHA-256: {n_ids:,} IDs -> {n_hashes:,} hashes (collision_free={n_ids == n_hashes})")

    return kept, discarded


# ── Transformed dataset ────────────────────────────────────────────────────

def build_transformed_dataset(df: pd.DataFrame) -> pd.DataFrame:
    """Add computed remaining lease and record ID to the clean dataset.

    Parameters
    ----------
    df : pd.DataFrame
        The clean dataset (before ID dedup).

    Returns
    -------
    pd.DataFrame
        A copy with ``remaining_lease_computed`` and ``rid`` columns added.
    """
    transformed = _compute_remaining_lease_columns(df.copy())
    transformed["rid"] = generate_record_ids(transformed)
    return transformed


# ── Failed records ──────────────────────────────────────────────────────────

def build_failed_dataset(
    key_dupes: pd.DataFrame,
    anomalies: pd.DataFrame,
    id_dupes: pd.DataFrame,
) -> pd.DataFrame:
    """Combine all rejected records into a single DataFrame with a reason tag.

    Parameters
    ----------
    key_dupes : pd.DataFrame
        Rows removed during composite-key deduplication.
    anomalies : pd.DataFrame
        Rows flagged by anomaly detection.
    id_dupes : pd.DataFrame
        Rows removed during record-ID deduplication.

    Returns
    -------
    pd.DataFrame
        Combined DataFrame with a ``failure_reason`` column.
    """
    tagged = []
    for reason, subset in [
        ("key_duplicate", key_dupes),
        ("anomaly", anomalies),
        ("id_duplicate", id_dupes),
    ]:
        chunk = subset.copy()
        chunk["failure_reason"] = reason
        tagged.append(chunk)

    combined = pd.concat(tagged, ignore_index=True)
    return combined


# ── Export all outputs ──────────────────────────────────────────────────────

def export_outputs(
    clean: pd.DataFrame,
    transformed: pd.DataFrame,
    failed: pd.DataFrame,
    hashed: pd.DataFrame,
    output_dir: Path,
) -> None:
    """Write all final datasets to CSV and print a summary.

    Parameters
    ----------
    clean : pd.DataFrame
        The cleaned dataset (post anomaly removal, pre ID assignment).
    transformed : pd.DataFrame
        Clean data with computed lease and record IDs.
    failed : pd.DataFrame
        All rejected records with failure reasons.
    hashed : pd.DataFrame
        Clean data with record IDs and SHA-256 hashes.
    output_dir : Path
        Directory to write CSV files into.
    """
    output_dir.mkdir(parents=True, exist_ok=True)

    datasets = [
        ("cleaned_dataset.csv", clean),
        ("transformed_dataset.csv", transformed),
        ("failed_records.csv", failed),
        ("hashed_dataset.csv", hashed),
    ]

    print(f"\n{SEPARATOR}")
    print("  Outputs")
    print(SEPARATOR)

    for filename, df in datasets:
        path = output_dir / filename
        df.to_csv(path, index=False)
        print(f"  {filename:<30} {len(df):>8,} rows")

    if "failure_reason" in failed.columns:
        print(f"\n  Failed record breakdown:")
        for reason, count in failed["failure_reason"].value_counts().items():
            print(f"    {reason:<20} {count:>8,}")

    print(f"\n  Output directory: {output_dir}")
    print(SEPARATOR)


# ── Apply ────────────────────────────────────────────────────────────────────

id_kept, id_discarded = deduplicate_by_record_id(clean_data)
transformed = build_transformed_dataset(clean_data)
failed = build_failed_dataset(deduped_fail, anomalous_data, id_discarded)
hashed = id_kept[["rid", "id_hash"]].join(
    clean_data, how="left"
)

export_outputs(
    clean=clean_data,
    transformed=transformed,
    failed=failed,
    hashed=id_kept,
    output_dir=CFG.manipulated_dir,
)

  Record IDs: 29,643 unique / 34,517 total
  Dedup: 29,643 kept, 4,874 discarded
  SHA-256: 29,643 IDs -> 29,643 hashes (collision_free=True)

-----------------------------------------------------------------
  Outputs
-----------------------------------------------------------------
  cleaned_dataset.csv              34,517 rows
  transformed_dataset.csv          34,517 rows
  failed_records.csv                7,510 rows
  hashed_dataset.csv               29,643 rows

  Failed record breakdown:
    id_duplicate            4,874
    anomaly                 2,166
    key_duplicate             470

  Output directory: /Users/john.yap/Desktop/DEA/manipulated_data
-----------------------------------------------------------------
